In [ ]:
import sys

import torch

sys.path.append(".")
import src.models.carsreid


MODELS_DIR = "./models"
LOGS_DIR = "./logs"

MODEL_RESUME = f"{LOGS_DIR}/epoch=6-step=99999.ckpt"

model = src.models.carsreid.CarsReidentificationTrainEffnet.load_from_checkpoint(MODEL_RESUME)

model = model.eval().half()

torch.save(model.net.state_dict(), f"{MODELS_DIR}/classifier.pt")

In [ ]:
import os

import tqdm
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize
import torch
from torch.utils.data import DataLoader
import torchvision.transforms.functional

import src.datasets.CarsDataset
import src.utils.transforms
import src.models.classifier

In [ ]:
EMBEDDING_DIM = 2048

DATASET_ROOT_DIR = "datasets/data"
DATABASE_ROOT = "./database"
SUBMITS_DIR = "./"

IMAGE_MEAN = [0.485, 0.456, 0.406]
IMAGE_STD = [0.229, 0.224, 0.225]
SUBMIT_NAME = "submission"

NEIGHBORS_COUNT_BATCH = 1024

NUM_CLASSES = 9630
params = {
        "model": "models/classifier.pt",
        "target_shape": (300, 300),
        "bs": 16,
        "version": "m",
        "flip": False,
        "use_db": False,
        "neightbors_num": 20,
        "weight": 1,
    }

# NUM_CLASSES = 5445
# params = {
#         "model": "models/embeddings_model.pt",
#         "target_shape": (300, 300),
#         "bs": 16,
#         "version": "m",
#         "flip": False,
#         "use_db": False,
#         "neightbors_num": 6,
#         "weight": 1,
#     }

In [ ]:
model = src.models.classifier.EffNetv2(
                class_num=NUM_CLASSES,
                features_dim=EMBEDDING_DIM,
                classify=True,
                mix_prec=True, 
                model_name=params['version']
            )
model.load_state_dict(torch.load(params["model"]))

model = model.eval()
model.classifier = model.classifier.float()
model.backbone = model.backbone.half()
model = model.cuda()

In [ ]:
def calc_score(emb1, emb2):
    score = torch.mm(emb1, emb2)
    return score.squeeze(1)


def get_embeddings(model, ds, loader, use_flip=False):
    embeddings = np.zeros((len(ds), EMBEDDING_DIM), dtype=np.float32)
    all_logits = np.zeros((len(ds), NUM_CLASSES), dtype=np.float32)

    files_dict = {}
    idx = 0
    for batch in tqdm.tqdm(loader, desc="Counting embeddings"):

        if isinstance(batch, (list, tuple)):
            batch = batch[0]  # Assume image is the first element (image, label)
            
        imgs = batch.float().cuda()
        if use_flip:
            for img_num in range(len(imgs)):
                imgs[img_num] = torchvision.transforms.functional.hflip(imgs[img_num])

        logits, features = model(imgs.half())

        features = features.detach().cpu().numpy()
        logits = logits.detach().cpu().numpy()

        embeddings[idx : idx + len(features)] = features
        all_logits[idx : idx + len(features)] = logits

        for _ in range(len(features)):
            files_dict[ds.imlist.loc[idx][0]] = idx
            idx = idx + 1

    return files_dict, embeddings, all_logits

In [ ]:
inf_transforms = src.utils.transforms.CarsTransforms(
    params["target_shape"], IMAGE_MEAN, IMAGE_STD, augs=False
)
inference_dataset = src.datasets.CarsDataset.StanfordCarsDataset(
        DATASET_ROOT_DIR, split="train", transforms=inf_transforms
    )
inf_loader = DataLoader(
    dataset=inference_dataset,
    batch_size=params["bs"],
    shuffle=False,
    num_workers=8,
    pin_memory=True,
    drop_last=False,
)

In [ ]:
import matplotlib.pyplot as plt
import torchvision
import torch

def show_image_grid(dataloader, n_images=16):
    """
    Plot a grid of images from a PyTorch dataloader.
    
    Args:
        dataloader: PyTorch DataLoader yielding images with shape [3, H, W]
        n_images: Total number of images to display in the grid
    """
    images_shown = 0

    for batch in dataloader:
        if isinstance(batch, (list, tuple)):
            images = batch[0]  # Assume image is the first element (image, label)
        else:
            images = batch

        # Limit to n_images
        images = images[:n_images]

        # Make a grid of images
        grid_img = torchvision.utils.make_grid(images, nrow=4, padding=2, normalize=True)

        plt.figure(figsize=(10, 10))
        plt.imshow(grid_img.permute(1, 2, 0))  # Convert [C, H, W] to [H, W, C]
        plt.axis('off')
        plt.title('Image Grid from DataLoader')
        plt.show()

        break  # Only one batch is needed

# show_image_grid(inf_loader)

In [ ]:
files_dict, embeddings, logits = get_embeddings(
    model, inference_dataset, inf_loader, use_flip=params["flip"]
)

In [ ]:
labels = [inference_dataset[i][1] for i in range(len(inference_dataset))]

In [ ]:
import numpy as np

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """
    Compute cosine similarity between two vectors.

    Args:
        a (np.ndarray): First embedding vector
        b (np.ndarray): Second embedding vector

    Returns:
        float: Cosine similarity score (-1 to 1)
    """
    a = a.flatten()
    b = b.flatten()

    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)

    if norm_a == 0 or norm_b == 0:
        return 0.0  # Avoid division by zero

    return dot_product / (norm_a * norm_b)

In [ ]:
nembeddings = normalize(embeddings)

In [ ]:
def load_db():
    db_embeddings = np.load(f"{DATABASE_ROOT}/db_embeddings.npy").astype(np.float32)
    db_embeddings = db_embeddings.reshape(db_embeddings.shape[0] * db_embeddings.shape[1], -1)
    
    db_labels = np.load(f"{DATABASE_ROOT}/db_labels.npy").astype(np.uint16)
    db_labels = np.repeat(db_labels, 5)  # т.к. эмбеддинги тоже размножены

    return db_embeddings, db_labels

db_embeddings, db_one_hot_labels = load_db()

In [ ]:
db_embeddings, db_one_hot_labels = load_db()

In [ ]:
# nembeddings = np.concatenate([nembeddings, db_embeddings], axis=0)
# logits = np.concatenate([logits, db_one_hot_labels], axis=0)

In [ ]:
tnembeddings = torch.from_numpy(nembeddings).float().cuda()
neighbors = torch.empty((len(tnembeddings), params["neightbors_num"]), dtype=torch.long)

for i in tqdm.tqdm(range(0, len(tnembeddings), NEIGHBORS_COUNT_BATCH), "Counting neighbors"):
    scores = torch.mm(tnembeddings, tnembeddings[i : i + NEIGHBORS_COUNT_BATCH].T)
    neighbors[i : i + NEIGHBORS_COUNT_BATCH] = torch.topk(
        scores, params["neightbors_num"], dim=0
    ).indices.T

In [ ]:
# nembeddings = normalize(embeddings)  # shape: [N_infer, D]
# ndb_embeddings = normalize(db_embeddings)  # shape: [N_db, D]

# # Convert to torch
# nembeddings_torch = torch.from_numpy(nembeddings).float().cuda()  # [N_infer, D]
# db_embeddings_torch = torch.from_numpy(ndb_embeddings).float().cuda()  # [N_db, D]

# # Compute cosine similarity: sim[i, j] = cos(embed_i, db_embed_j)
# # Using matrix multiplication (dot product since both are L2-normalized)
# similarity_matrix = torch.mm(nembeddings_torch, db_embeddings_torch.T)  # [N_infer, N_db]

# # Optional: move to CPU and convert to numpy
# topk_vals, topk_indices = similarity_matrix.topk(k=5, dim=1, largest=True, sorted=True)
# topk_labels = db_one_hot_labels[topk_indices.cpu().numpy()]

import json

with open('datasets/class_id_to_label.json', 'r') as f:
    le = json.load(f)

preds = np.argmax(logits, axis=1)

In [ ]:
def plot_one(element, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    img = element[0].permute(1, 2, 0).numpy()

    if mean is not None and std is not None:
        mean = np.array(mean)
        std = np.array(std)
        img = img * std + mean  # de-normalize

    img = np.clip(img, 0, 1)
    
    plt.imshow(img)
    plt.show()

In [ ]:
num = 8000
plot_one(inference_dataset[num])

for i, val in enumerate(topk_labels[num]):
    print('Predicted Model Name:', le[str(val)], topk_vals[num][i].item())

In [ ]:
preds = np.argmax(logits, axis=1)

In [ ]:
num = 3500

for i in neighbors[num]:
    try:
        print('Stanford label:', labels[i], 'Predicted Model Name:', le[str(preds[i])], cosine_similarity(nembeddings[neighbors[num][0].numpy()], nembeddings[i.numpy()]))

        plot_one(inference_dataset[i.numpy()])
    except:
        print('Predicted Model Name:', le[str(preds[i])], cosine_similarity(nembeddings[neighbors[num][0].numpy()], nembeddings[i.numpy()]))

In [ ]:
# Suppose:
# topk_indices: shape [8144, 6] — index 0 is query, indices 1:6 are top-5 retrieved
# labels: list or array of shape [8144], where labels[i] is class of embedding i

def compute_acc_map(topk_indices, labels, k=5):
    topk_indices = np.array(topk_indices)
    labels = np.array(labels)

    correct_at_1 = 0
    average_precisions = []

    for row in topk_indices:
        query_idx = row[0]
        retrieved_idxs = row[1:k+1]

        query_label = labels[query_idx]
        retrieved_labels = labels[retrieved_idxs]

        # Accuracy@1
        if retrieved_labels[0] == query_label:
            correct_at_1 += 1

        # AP@5
        num_correct = 0
        precision_sum = 0.0
        for rank, label in enumerate(retrieved_labels):
            if label == query_label:
                num_correct += 1
                precision = num_correct / (rank + 1)
                precision_sum += precision

        correct_matches = np.sum(retrieved_labels == query_label)
        if correct_matches > 0:
            ap = precision_sum / min(k, correct_matches)
        else:
            ap = 0.0

        average_precisions.append(ap)

    acc1 = correct_at_1 / len(topk_indices)
    map5 = np.mean(average_precisions)

    return acc1, map5

# Example usage
acc1, map5 = compute_acc_map(neighbors, labels, k=5)
print(f"Accuracy@1: {acc1:.4f}, mAP@5: {map5:.4f}")

In [ ]:
def correct_scores_with_logits(neighbors, logits, embeddings, labels, k=5):
    """
    Returns:
        acc@1, map@5, new_neighbors — reordered neighbors based on confidence-adjusted score
    """
    embeddings = normalize(embeddings)
    logits = logits.copy()

    num_samples = embeddings.shape[0]
    logits_exp = np.exp(logits)
    softmax_logits = logits_exp / np.sum(logits_exp, axis=1, keepdims=True)

    acc1 = 0
    map5_total = []
    new_neighbors = np.zeros((num_samples, k), dtype=np.int32)

    for idx in tqdm.tqdm(range(num_samples), desc="Correcting scores with logits"):
        query_label = labels[idx]
        neighbor_indices = neighbors[idx][1:k+1]  # skip self (0)

        # Softmax logits
        cluster_logits = softmax_logits[neighbor_indices]
        cluster_classes = cluster_logits.argmax(axis=1)

        cluster_logits_query = softmax_logits[idx]
        cluster_class_query = cluster_logits_query.argmax()

        cluster_class_conf = cluster_logits[np.arange(len(neighbor_indices)), cluster_classes].mean()
        query_class_conf = cluster_logits_query[cluster_class_query]

        # Reweighting confidence
        cluster_is_query_conf = 1 - (
            cluster_logits[np.arange(len(neighbor_indices)), cluster_classes] -
            cluster_logits[np.arange(len(neighbor_indices)), cluster_class_query] + 1e-5
        ) / (
            cluster_logits[np.arange(len(neighbor_indices)), cluster_classes] +
            cluster_logits[np.arange(len(neighbor_indices)), cluster_class_query] + 1e-5
        )

        avg_conf = cluster_is_query_conf.mean()
        trust = min(query_class_conf, cluster_class_conf)

        # Corrected similarity score
        sims = np.dot(embeddings[idx], embeddings[neighbor_indices].T)
        corrected_scores = trust * avg_conf + (1 - trust) * sims

        # Sort neighbors by corrected scores
        sorted_idx = np.argsort(corrected_scores)[::-1]
        reordered_indices = neighbor_indices[sorted_idx]
        new_neighbors[idx] = reordered_indices

        final_neighbor_labels = labels[reordered_indices]

        # Accuracy@1
        acc1 += (final_neighbor_labels[0] == query_label)

        # mAP@5
        num_correct = 0
        ap_sum = 0
        for rank, lbl in enumerate(final_neighbor_labels):
            if lbl == query_label:
                num_correct += 1
                ap_sum += num_correct / (rank + 1)
        ap = ap_sum / min(k, np.sum(final_neighbor_labels == query_label)) if num_correct > 0 else 0
        map5_total.append(ap)

    acc1 /= num_samples
    map5 = np.mean(map5_total)

    return acc1, map5, new_neighbors

In [ ]:
acc1, map5, corrected_neighbors = correct_scores_with_logits(
    neighbors.cpu().numpy(), logits, embeddings, np.array(labels), k=5
)

print(f"Corrected Accuracy@1: {acc1:.4f}, mAP@5: {map5:.4f}")